# So sánh hiệu năng và chất lượng ASR: 6 Mô hình / Cấu hình trên GPU

Notebook này thực hiện nhận diện giọng nói Tiếng Việt và so sánh giữa 5 cấu hình:
1. **faster-whisper-medium** (CTranslate2 FP16)
2. **wav2vec 2.0** (`khanhld/wav2vec2-base-vietnamese-160h` FP32, raw CTC)
3. **faster-whisper-large-v3 (INT8)** (`large-v3` với CTranslate2 INT8 lượng tử hóa)
4. **Wav2Vec 2.0 + Punctuation Model** (`wav2vec2` kết hợp `dragonSwing/vibert-capu` phục hồi dấu câu & viết hoa)
5. **NVIDIA FastConformer** (`nvidia/parakeet-ctc-0.6b-vi` qua NeMo toolkit)

In [ ]:
import os
import gc
import sys
import time
import json
import tempfile
import torch
import soundfile as sf
from pathlib import Path
from tqdm import tqdm
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

# Thêm đường dẫn capu vào python path để load GecBERTModel
sys.path.append(str(Path("capu").resolve()))

# Kiểm tra GPU CUDA
print("--- KIỂM TRA THIẾT BỊ GPU ---")
print("PyTorch CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("PyTorch GPU Device:", torch.cuda.get_device_name(0))


In [ ]:
ROOT = Path(".")
AUDIO_PATH = Path("../data/audio")
OUTPUT_PATH = ROOT / "asr_comparison_results.json"


### Định nghĩa các hàm phụ trợ nhận diện

In [ ]:
def run_faster_whisper(model_obj, wav_path):
    segments, info = model_obj.transcribe(str(wav_path), language="vi", beam_size=5)
    results = []
    for s in segments:
        results.append({
            "start": round(s.start, 2),
            "end": round(s.end, 2),
            "text": s.text.strip()
        })
    return results

def run_wav2vec2(pipeline_obj, wav_path):
    audio, sr = sf.read(str(wav_path))
    res = pipeline_obj(audio, chunk_length_s=30, stride_length_s=5, return_timestamps="word")
    
    words = res.get("chunks", [])
    segments = []
    current_segment_words = []
    start_time = None
    
    for w in words:
        w_text = w.get("text", "")
        w_ts = w.get("timestamp", (None, None))
        if w_ts is None or w_ts[0] is None or w_ts[1] is None:
            continue
        w_start, w_end = w_ts
        if start_time is None:
            start_time = w_start
            
        if current_segment_words and (w_start - current_segment_words[-1]["end"] > 1.5 or len(current_segment_words) >= 15):
            segments.append({
                "start": round(start_time, 2),
                "end": round(current_segment_words[-1]["end"], 2),
                "text": " ".join([x["text"] for x in current_segment_words])
            })
            current_segment_words = []
            start_time = w_start
            
        current_segment_words.append({"text": w_text, "end": w_end})
        
    if current_segment_words:
        segments.append({
            "start": round(start_time, 2),
            "end": round(current_segment_words[-1]["end"], 2),
            "text": " ".join([x["text"] for x in current_segment_words])
        })
    return segments

def run_wav2vec2_with_punc(model_obj, wav2vec_segments):
    results = []
    for seg in wav2vec_segments:
        raw_text = seg["text"]
        if not raw_text.strip():
            punc_text = ""
        else:
            try:
                punc_res = model_obj(raw_text)
                punc_text = " ".join(punc_res).strip()
            except Exception as e:
                punc_text = raw_text
        results.append({
            "start": seg["start"],
            "end": seg["end"],
            "text": punc_text
        })
    return results

def run_nemo_fastconformer(model_obj, wav_path):
    audio, sr = sf.read(str(wav_path))
    chunk_size = 30 * sr
    results = []
    
    for i in range(0, len(audio), chunk_size):
        chunk = audio[i : i + chunk_size]
        if len(chunk) < 1.0 * sr:
            continue
        start_time = i / sr
        end_time = min((i + chunk_size) / sr, len(audio) / sr)
        
        with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as temp_wav:
            sf.write(temp_wav.name, chunk, sr)
            temp_name = temp_wav.name
            
        try:
            res = model_obj.transcribe([temp_name])
            text = res[0] if isinstance(res[0], str) else res[0].text
            text = text.strip()
            if text:
                results.append({
                    "start": round(start_time, 2),
                    "end": round(end_time, 2),
                    "text": text
                })
        finally:
            try:
                os.remove(temp_name)
            except:
                pass
    return results


### Đánh giá tuần tự từng Mô hình (Để tránh tràn VRAM GPU)

In [ ]:
wav_files = sorted(list(AUDIO_PATH.glob("*.wav")))
print(f"Tìm thấy {len(wav_files)} tệp âm thanh để thử nghiệm.")

results_dict = {f.stem: {"audio_key": f.stem} for f in wav_files}

# ========================================
# 1. Chạy faster-whisper-medium (FP16)
# ========================================
print("\n--- 1. Khởi chạy faster-whisper-medium (FP16) ---")
from faster_whisper import WhisperModel
whisper_medium = WhisperModel("medium", device="cuda", compute_type="float16")

for wav_file in wav_files:
    start_t = time.time()
    try:
        res = run_faster_whisper(whisper_medium, wav_file)
    except Exception as e:
        res = [{"start": 0.0, "end": 0.0, "text": f"[Lỗi: {str(e)}]"}]
    time_t = time.time() - start_t
    results_dict[wav_file.stem]["faster_whisper_medium_segments"] = res
    results_dict[wav_file.stem]["faster_whisper_medium_time"] = time_t
    print(f"- {wav_file.name} hoàn thành trong: {time_t:.2f}s")

del whisper_medium
torch.cuda.empty_cache()
gc.collect()

# ========================================
# 2. Chạy wav2vec 2.0 & 5. Wav2Vec 2.0 + Punctuation
# ========================================
print("\n--- 2. Khởi chạy wav2vec 2.0 (vietnamese base) ---")
from transformers import pipeline
wav2vec_pipeline = pipeline(
    "automatic-speech-recognition",
    model="khanhld/wav2vec2-base-vietnamese-160h",
    device=0
)

for wav_file in wav_files:
    start_t = time.time()
    try:
        res = run_wav2vec2(wav2vec_pipeline, wav_file)
    except Exception as e:
        res = [{"start": 0.0, "end": 0.0, "text": f"[Lỗi: {str(e)}]"}]
    time_t = time.time() - start_t
    results_dict[wav_file.stem]["wav2vec2_segments"] = res
    results_dict[wav_file.stem]["wav2vec2_time"] = time_t
    print(f"- {wav_file.name} (raw) hoàn thành trong: {time_t:.2f}s")

del wav2vec_pipeline
torch.cuda.empty_cache()
gc.collect()

print("\n--- 5. Khởi chạy Punctuation Model (vibert-capu) ---")
from gec_model import GecBERTModel
punc_model = GecBERTModel(
    vocab_path="capu/vocabulary",
    model_paths="capu",
    split_chunk=True
)

for wav_file in wav_files:
    start_t = time.time()
    raw_segs = results_dict[wav_file.stem]["wav2vec2_segments"]
    try:
        res = run_wav2vec2_with_punc(punc_model, raw_segs)
    except Exception as e:
        res = [{"start": 0.0, "end": 0.0, "text": f"[Lỗi: {str(e)}]"}]
    time_t = time.time() - start_t + results_dict[wav_file.stem]["wav2vec2_time"]
    results_dict[wav_file.stem]["wav2vec2_punctuation_segments"] = res
    results_dict[wav_file.stem]["wav2vec2_punctuation_time"] = time_t
    print(f"- {wav_file.name} (+ Punc) hoàn thành trong: {time_t:.2f}s")

del punc_model
torch.cuda.empty_cache()
gc.collect()

# ========================================
# 4. Chạy faster-whisper-large-v3 (INT8)
# ========================================
print("\n--- 4. Khởi chạy faster-whisper-large-v3 (INT8) ---")
whisper_large_int8 = WhisperModel("large-v3", device="cuda", compute_type="int8_float16")

for wav_file in wav_files:
    start_t = time.time()
    try:
        res = run_faster_whisper(whisper_large_int8, wav_file)
    except Exception as e:
        res = [{"start": 0.0, "end": 0.0, "text": f"[Lỗi: {str(e)}]"}]
    time_t = time.time() - start_t
    results_dict[wav_file.stem]["faster_whisper_large_int8_segments"] = res
    results_dict[wav_file.stem]["faster_whisper_large_int8_time"] = time_t
    print(f"- {wav_file.name} hoàn thành trong: {time_t:.2f}s")

del whisper_large_int8
torch.cuda.empty_cache()
gc.collect()

# ========================================
# 6. Chạy NVIDIA FastConformer (parakeet-ctc-0.6b-vi)
# ========================================
print("\n--- 6. Khởi chạy NVIDIA FastConformer ---")
import nemo.collections.asr as nemo_asr
nemo_model = nemo_asr.models.ASRModel.from_pretrained("nvidia/parakeet-ctc-0.6b-vi")
nemo_model = nemo_model.to("cuda")

for wav_file in wav_files:
    start_t = time.time()
    try:
        res = run_nemo_fastconformer(nemo_model, wav_file)
    except Exception as e:
        res = [{"start": 0.0, "end": 0.0, "text": f"[Lỗi: {str(e)}]"}]
    time_t = time.time() - start_t
    results_dict[wav_file.stem]["nemo_fastconformer_segments"] = res
    results_dict[wav_file.stem]["nemo_fastconformer_time"] = time_t
    print(f"- {wav_file.name} hoàn thành trong: {time_t:.2f}s")

del nemo_model
torch.cuda.empty_cache()
gc.collect()

# ========================================
# Ghi kết quả dạng JSON
# ========================================
results_log = list(results_dict.values())
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(results_log, f, ensure_ascii=False, indent=4)

print(f"========================================\nĐã lưu kết quả tại: {OUTPUT_PATH}")


### In ra tổng thời gian xử lý và so sánh

In [ ]:
total_whisper_time = sum(x["faster_whisper_medium_time"] for x in results_log)
total_wav2vec2_time = sum(x["wav2vec2_time"] for x in results_log)
total_large_time = sum(x["faster_whisper_large_int8_time"] for x in results_log)
total_punc_time = sum(x["wav2vec2_punctuation_time"] for x in results_log)
total_nemo_time = sum(x["nemo_fastconformer_time"] for x in results_log)

print("Tổng thời gian chạy trên GPU cho cả 2 tệp WAV:")
print(f"- 1. faster-whisper-medium: {total_whisper_time:.2f} s")
print(f"- 2. wav2vec 2.0 (raw CTC): {total_wav2vec2_time:.2f} s")
print(f"- 3. faster-whisper-large-v3 (INT8): {total_large_time:.2f} s")
print(f"- 4. Wav2Vec 2.0 + Punctuation: {total_punc_time:.2f} s")
print(f"- 5. NVIDIA FastConformer: {total_nemo_time:.2f} s")
